# QFabric — Overview & Run Order

**Quantum Network Emulation on FABRIC: BB84 QKD over a P4 programmable data plane.**

Alice (photon source) → BMv2 P4 switch (fiber-loss quantum channel, EtherType `0x7101`) → Bob (detector).
The classical channel rides the **same switch** as raw `0x7102` frames (TCP is a dev fallback). Slices are
single-site by default: photons cannot cross a WAN, so *distance is emulated* by one knob that drives both
fiber loss and classical propagation delay; cross-site / netem runs are explicit stress studies.

## Run the notebooks in this order

**Every notebook after this one runs on a FABRIC slice** — this one is just orientation and an
environment check, so it needs no slice and no FABRIC account. The rest are grouped into folders by
workflow and numbered *within* each folder, so a notebook is always named by its folder: `fabric/02`,
`concepts/05`. Start with `fabric/01` — the other two tracks assume the slice it builds.

### `fabric/` — deploy, run, analyse
| # | Notebook | What it does |
|---|----------|--------------|
| 01 | [`fabric/01_setup_slice`](fabric/01_setup_slice.ipynb) | Provision the slice, install BMv2, compile P4, start the switch |
| 02 | [`fabric/02_run_experiment`](fabric/02_run_experiment.ipynb) | Run BB84 across the slice, collect results, verify |
| 03 | [`fabric/03_analysis`](fabric/03_analysis.ipynb) | Load what `02` recorded and generate the plots & tables |
| 04 | [`fabric/04_all_scenarios`](fabric/04_all_scenarios.ipynb) | **Every** scenario (singles + sweeps) with 4-way cross-validation (QFabric measured / QFabric-sim / SeQUeNCe / NetSquid) + sweep figures |
| 05 | [`fabric/05_network_effects`](fabric/05_network_effects.ipynb) | Classical-network (latency/jitter/loss) impact on QKD throughput — a stress study |

### `sequence/` — the distributed SeQUeNCe runtime
| # | Notebook | What it does |
|---|----------|--------------|
| 01 | [`sequence/01_emulator`](sequence/01_emulator.ipynb) | Distributed **SeQUeNCe** BB84 over the P4 path, with the timeline certificate |
| 02 | [`sequence/02_scenarios`](sequence/02_scenarios.ipynb) | Scenario sweeps of the distributed emulator vs the NetSquid reference |
| 03 | [`sequence/03_entanglement_e91`](sequence/03_entanglement_e91.ipynb) | Entanglement-based QKD (**E91 / BBM92**); CHSH Bell test across nodes |

### `concepts/` — one capability each, measured on the slice
| # | Notebook | What it does |
|---|----------|--------------|
| 01 | [`concepts/01_eavesdropper`](concepts/01_eavesdropper.ipynb) | Intercept-resend attack: QBER vs Eve's tap fraction, the ~11% threshold |
| 02 | [`concepts/02_reconciliation`](concepts/02_reconciliation.ipynb) | Cascade reconciliation → Alice's and Bob's keys match bit-for-bit |
| 03 | [`concepts/03_repeater`](concepts/03_repeater.ipynb) | Entanglement swapping across three real nodes (Werner law, CHSH vs hops) |
| 04 | [`concepts/04_qkd_security`](concepts/04_qkd_security.ipynb) | Finite-key bounds, authenticated channel, biased bases, live decoy |
| 05 | [`concepts/05_distributed_computing`](concepts/05_distributed_computing.ipynb) | A **gate** between two QPUs: teleport / non-local CNOT, and what a lost correction costs |

Tracks: **deploy** `fabric/01` → `02` → `04` → `03`. **Learn QKD** `fabric/02` → `concepts/01` → `02` → `04`.
**Entanglement & compute** `sequence/03` → `concepts/03` → `concepts/05`.

## Prerequisites
- A FABRIC account, a project, and your tokens configured in JupyterHub — needed by every notebook
  *except this one*.
- `pip install -r requirements.txt` from the project root (Python 3.11 recommended).
- For the cross-validation (`fabric/04`): netsquid.org credentials in `NETSQUID_USER` / `NETSQUID_PASS`;
  it builds the SeQUeNCe / NetSquid / QFabric-sim envs on the slice nodes via `deploy_fabric.setup_sim_envs`.
  See the README.

### At a glance
- **Purpose:** orient you and confirm your environment before touching a slice.
- **Inputs:** none.
- **Outputs:** an environment report saying which dependencies are present *here* — and which belong on the slice nodes instead.
- **Runs on / runtime:** anywhere; seconds. The only notebook that does not need a slice.
- **If something fails:** the check prints the install command for whatever is missing (`pip install -e ".[dev]"` for core, `".[fabric]"` for fablib). Python 3.11 recommended.

## Environment check

Confirms the dependencies this notebook and the slice notebooks need. It only checks that
things **import** — to confirm the library actually computes correctly before you spend
slice time, run the test suite from the project root:

```
pip install -e ".[dev]" && pytest tests/ -q
```

In [ ]:
import sys, importlib
from pathlib import Path

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'qne').is_dir())
sys.path.insert(0, str(PROJECT_DIR))

def check(mod, required=False):
    try:
        m = importlib.import_module(mod)
        print(f'  [OK]       {mod:26s} {getattr(m, "__version__", "")}')
        return True
    except Exception as e:
        print(f'  {"[MISSING]" if required else "[optional]":10s} {mod:26s} ({e.__class__.__name__})')
        return False

print(f'Python {sys.version.split()[0]}  |  project: {PROJECT_DIR}\n')

print('Required by this notebook:')
core_ok = all([check('numpy', True), check('yaml', True),
               check('qne', True), check('validation', True)])

print('\nRequired by every notebook after this one (they all call get_fablib):')
fablib_ok = check('fabrictestbed_extensions', True)

print()
if not core_ok:
    print('Install core deps:  pip install -e ".[dev]"')
elif not fablib_ok:
    print('Core OK. Install fablib before the slice notebooks:  pip install -e ".[fabric]"')
    print('(On FABRIC JupyterHub it is already present.)')
else:
    print('Environment OK — continue to fabric/01_setup_slice.')

# Deliberately not checked here, because none of it belongs in this kernel:
#   * SeQUeNCe / NetSquid for the on-slice cross-validation (notebook fabric/04)
#     are installed ON the slice nodes by deploy_fabric.setup_sim_envs.
#   * The qne-sequence runtime is built on the nodes by setup_sequence_runtime.
#   * qiskit backs QiskitRegister, used by the test suite and
#     dqc.run_dqc_session(backend='qiskit'); no notebook needs it.

---
**Next:** `fabric/01_setup_slice` to provision the FABRIC slice.